In [1]:
import pandas as pd
import numpy as np

In [2]:
n = 200

df = pd.DataFrame({

    "account_id": range(n),

    "symbol": np.random.choice(
        ["BTC", "ETH", "SOL"],
        n
    ),

    "price": np.random.randint(
        100,
        50000,
        n
    ),

    "position": np.random.randint(
        1,
        200,
        n
    ),

    "balance": np.random.randint(
        1000,
        50000,
        n
    ),

    "leverage": np.random.choice(
        [5, 10, 20, 50, 100],
        n
    )

})  #创建模拟账户

In [3]:
df["notional"] = df["price"] * df["position"]  #notional

In [4]:
def mm_rate(x):

    if x < 100000:
        return 0.05

    elif x < 500000:
        return 0.1

    elif x < 1000000:
        return 0.15

    else:
        return 0.2


df["mm_rate"] = df["notional"].apply(mm_rate)  #MM_rate

In [5]:
df["maintenance_margin"] = (
    df["notional"] * df["mm_rate"]
)

In [6]:
df["initial_margin"] = (
    df["notional"] / df["leverage"]
)

In [7]:
df["margin_ratio"] = (
    df["balance"] / df["notional"]
)

In [8]:
df["liquidation"] = (
    df["balance"]
    < df["maintenance_margin"]
).astype(int)

df["liquidation"].value_counts()

,count
liquidation,
1,172
0,28


In [9]:
big_pos = df[
    df["notional"] > 500000
]

big_pos.head()

,account_id,symbol,price,position,balance,leverage,notional,mm_rate,maintenance_margin,initial_margin,margin_ratio,liquidation
2,2,ETH,14497,109,41796,100,1580173,0.20,316034.60,15801.73,0.026450,1
3,3,BTC,30684,100,47352,100,3068400,0.20,613680.00,30684.00,0.015432,1
4,4,SOL,9911,88,7501,10,872168,0.15,130825.20,87216.80,0.008600,1
5,5,BTC,33897,107,22508,5,3626979,0.20,725395.80,725395.80,0.006206,1
6,6,ETH,3389,149,45926,20,504961,0.15,75744.15,25248.05,0.090950,1


In [10]:
shock = 0.7

df["shock_price"] = df["price"] * shock

df["shock_notional"] = (
    df["shock_price"] * df["position"]
)

In [11]:
df["shock_mm"] = (
    df["shock_notional"]
    * df["mm_rate"]
)

In [12]:
df["shock_liq"] = (
    df["balance"]
    < df["shock_mm"]
).astype(int)

df["shock_liq"].value_counts() #shock liquidation

,count
shock_liq,
1,170
0,30


In [13]:
liq_loss = df[
    df["shock_liq"] == 1
]["shock_notional"].sum()

liq_loss

np.float64(365044340.49999994)

In [14]:
insurance_fund = 5_000_000

adl = liq_loss > insurance_fund

adl

np.True_

In [15]:
print("Total accounts:", len(df))

print("Liquidations:", df["liquidation"].sum())

print("Shock liquidations:", df["shock_liq"].sum())

print("Loss:", liq_loss)

print("ADL triggered:", adl)

Total accounts: 200
Liquidations: 172
Shock liquidations: 170
Loss: 365044340.49999994
ADL triggered: True
